# 07 · Health access under disruption


**Mission (35 minutes):** compare nearest-clinic distance before and after a closure, then test travel-speed assumptions. Deliver a service-access gap map. All clinics and capacities are fictional; great-circle travel times are lower-bound illustrations, not routes or dispatch advice.


**Runtime:** Python + NumPy + Matplotlib; no model key. Run all cells in order. Interactive viewers additionally load JavaScript from a CDN. All scenario records are fictional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gev_lab import *
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
scenario = styled_sectors()
stats = sector_summary()
print('SYNTHETIC TRAINING EXERCISE | snapshot', AS_OF)

In [ ]:
clinics = [f for f in scenario['features'] if f['properties']['kind']=='clinic']
xy = np.array([[s['lon'],s['lat']] for s in stats])
cxy = np.array([c['geometry']['coordinates'] for c in clinics])
dist = haversine_km(xy[:,0,None],xy[:,1,None],cxy[None,:,0],cxy[None,:,1])
open_mask = np.array([c['properties']['available'] for c in clinics])
baseline = dist.min(axis=1)
disrupted = np.where(open_mask[None,:],dist,np.inf).min(axis=1)
assert np.all(disrupted >= baseline)
fig, axes = plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
for ax, values, title in zip(axes,[baseline,disrupted-baseline],['All clinics available','Added distance after C3 closure']):
    p=plot_sectors(ax,values,title,cmap='YlGnBu');fig.colorbar(p,ax=ax,shrink=.75,label='Great-circle km')
plt.show()

## Travel-time assumptions change the story
Use two invented effective speeds. These omit roads, river crossings, terrain, security, queues, and referral capability. A 60-minute threshold is chosen for the exercise, not a clinical standard. A centroid is not the location of every resident.

In [ ]:
pop=np.array([s['population'] for s in stats])
speeds=np.array([5,15,30,45])
share=np.array([pop[disrupted/speed*60 > 60].sum()/pop.sum()*100 for speed in speeds])
fig,ax=plt.subplots(figsize=(8,4),constrained_layout=True)
ax.plot(speeds,share,'o-',color='#128d87');ax.set(xlabel='Assumed effective speed (km/h)',ylabel='Population beyond 60 min (%)',ylim=(0,100),title='Synthetic access sensitivity · straight-line approximation');ax.grid(alpha=.2);plt.show()
assignment=np.where(open_mask[None,:],dist,np.inf).argmin(axis=1)
for j,c in enumerate(clinics):
    print(c['id'],'assigned population',pop[assignment==j].sum(),'available',open_mask[j])

## Capacity is a separate constraint
Nearest-facility assignment can overwhelm the closest clinic. This one-day demand fraction is an invented workload scenario, unrelated to the cumulative case count.

In [ ]:
demand_fraction=.02
for j,c in enumerate(clinics):
    demand=pop[assignment==j].sum()*demand_fraction
    capacity=c['properties']['capacity_day'] if open_mask[j] else 0
    print(f"{c['id']}: scenario visits/day {demand:.0f}; capacity {capacity}; unmet {max(0,demand-capacity):.0f}")

## Lab challenge
Restore C3 and compare accessibility and capacity. Then close all clinics: represent lack of service as infinity/missing, not zero distance, and avoid assigning people to a closed clinic. Discuss the extra evidence needed for a road/boat network model.

**Check:** closing a clinic cannot reduce nearest-open-clinic distance. Population, demand, capacity, and physical access are different quantities. A distance map cannot establish safe access or service quality.

**Sources:** [WHO AccessMod](https://www.who.int/tools/accessmod-geographic-access-to-health-care) for a real accessibility modeling framework; [GeoJSON coordinate convention, RFC 7946](https://www.rfc-editor.org/rfc/rfc7946). This lab does not implement AccessMod or use its datasets.